# Evidence recall và hai hướng giảm rơi bằng chứng (VimQA n=500, Qwen2.5-7B 4-bit)

Trả lời góp ý reviewer (camera-ready SOICT): bộ nén giữ được bao nhiêu câu bằng chứng vàng, và hai hướng có làm rơi ít hơn không.

| Run | Cấu hình | Thời gian ước tính (T4) |
|---|---|---|
| E0 | IterCOMP-X mặc định (đối chứng ghép cặp) | ~45 phút |
| E1 | `--window 1`: kéo theo câu kề ±1 cùng tài liệu | ~50 phút |
| E2 | `--followup-mode entity --title-anchor`: câu hỏi phụ gọi đích danh thực thể + khớp tiêu đề | ~50 phút |
| E3 | `exp2_undiacritised.py --modes keep,strip`: recall khi bỏ dấu | ~1,5 giờ |

Cả bốn đều ghi các câu được chọn `(doc_title, sent_id)`, nên recall đo trên ngữ cảnh đầy đủ (bản cũ cắt ngữ cảnh ở 400 ký tự nên không đo được).
Cần: **Accelerator = GPU T4 x2**, **Internet = On**. Mỗi run có checkpoint; chạy lại thì bỏ qua run đã xong.

In [ ]:
# ══ CỬA CHẶN: GPU có chạy được bitsandbytes 4-bit không? ══
# Kaggle cấp NGẪU NHIÊN P100 (sm_60) hoặc T4 (sm_75) nếu metadata không ghi rõ
# machine_shape. PyTorch của Kaggle chỉ build cho sm_70 trở lên, nên trên P100
# bitsandbytes chết bằng SIGSEGV giữa lúc nạp trọng số:
#
#     Error named symbol not found at line 74 in file /src/csrc/ops.cu
#     rc=-11
#
# Lỗi đó mất ~2 phút mới hiện và thông báo không nói gì về nguyên nhân. Ô này
# phát hiện trong 5 giây và nói thẳng phải làm gì.
import torch

assert torch.cuda.is_available(), (
    'Không có GPU. Settings > Accelerator > GPU T4 x2, rồi chạy lại.')

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name}  sm_{major}{minor}  {gb:.1f} GB')

if major < 7:
    raise SystemExit(
        f'\n{name} là sm_{major}{minor} - PyTorch của Kaggle không hỗ trợ.\n'
        'Cần T4 (sm_75). Hai cách:\n'
        '  1. Settings > Accelerator > chọn "GPU T4 x2" (không phải "GPU P100")\n'
        '  2. Nếu đẩy bằng CLI: thêm "machine_shape": "NvidiaTeslaT4" vào\n'
        '     kernel-metadata.json\n'
        'Chạy tiếp trên P100 sẽ SIGSEGV lúc nạp mô hình 4-bit.')

print('✓ GPU chạy được bitsandbytes 4-bit')

In [ ]:
# ══ GHIM transformers VỀ 4.x - ĐÂY LÀ BẢN VÁ THẬT ══
# Kaggle nay ship transformers 5.0.0. llmlingua 0.2.2 viết cho 4.x, và hai bên
# đòi hai thứ LOẠI TRỪ NHAU cho cùng một object `past_key_values`:
#
#   transformers 5.0  ->  phải là Cache, gọi .get_seq_length()
#   llmlingua 0.2.2   ->  phải là list, lặp `for k, v in past_key_values`
#
# Không monkey-patch nào làm hài lòng cả hai, vì cùng một object đi qua cả hai
# nơi. SÁU lần vá đều chết vì cố làm điều bất khả. Đường LongLLMLingua đã được
# chạy thử THÀNH CÔNG trên 4.45.2 ở máy dev, nên ghim về đúng bản đó cũng khép
# luôn lỗ hổng "chạy được ở đây, chết trên Kaggle".
!pip install -q "transformers==4.45.2" llmlingua rank_bm25 tiktoken python-dotenv bitsandbytes accelerate 2>&1 | tail -3

import transformers
print('transformers =', transformers.__version__)
assert transformers.__version__.startswith('4.'), (
    f'Ghim KHÔNG ăn: đang là {transformers.__version__}. Nếu transformers đã '
    f'được import trước khi pip chạy thì phải Restart & Run All.')


In [ ]:
# Lấy mã nguồn bằng cách clone repo. Trước đây repo còn private nên notebook
# phải tải qua một gist công khai; repo public rồi, nên clone thẳng là nguồn duy
# nhất, không còn hai bản phải giữ đồng bộ.
import os, sys, subprocess, shutil

BASE = ('/content' if os.path.isdir('/content')
        else '/kaggle/working' if os.path.isdir('/kaggle/working')
        else os.getcwd())
REPO = os.path.join(BASE, 'ANLP-IterCOMP')
GIT_URL = 'https://github.com/nhantrnh/ANLP-IterCOMP.git'
# 'budget' nằm trong danh sách này từ một bản cũ; module đó không tồn tại ở
# repo nào, nên cửa chặn luôn báo thiếu và kernel chết trước khi chạy gì.
NEED = ['__init__', 'core', 'scorer', 'llm', 'reader', 'data', 'metrics', 'fertility', 'stats', 'stopping']

os.chdir(BASE)
shutil.rmtree(REPO, ignore_errors=True)
for m in list(sys.modules):
    if m == 'itercomp' or m.startswith('itercomp.'):
        del sys.modules[m]

subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, REPO], check=True)

miss = [m for m in NEED
        if not os.path.exists(os.path.join(REPO, 'src', 'itercomp', m + '.py'))]
if miss:
    raise SystemExit('Bản clone thiếu module: ' + ', '.join(miss))

os.chdir(REPO)
if os.path.join(REPO, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(REPO, 'src'))

sha = subprocess.run(['git', 'log', '-1', '--format=%h %s'],
                     capture_output=True, text=True).stdout.strip()
print('cwd:', os.getcwd(), '| src/scripts/:', os.path.isdir('src/scripts'))
print('đủ', len(NEED), 'module | commit:', sha)


In [ ]:
import urllib.request
p = 'data/vimqa/validation.parquet'
os.makedirs(os.path.dirname(p), exist_ok=True)
if not os.path.exists(p):
    urllib.request.urlretrieve('https://huggingface.co/datasets/nguyenlab/vimqa/resolve/main/'
                               'data/validation-00000-of-00001.parquet', p)
# Cờ mới phải có trong bản clone; thiếu thì repo chưa được push.
s = open('src/scripts/run_eval.py', encoding='utf-8').read()
assert '--window' in s and '--title-anchor' in s, 'Repo chưa có --window/--title-anchor: push src/ trước.'
assert '--modes' in open('src/scripts/exp2_undiacritised.py', encoding='utf-8').read(), 'exp2 chưa có --modes'
print('✓ dữ liệu và mã sẵn sàng')

In [ ]:
import subprocess, sys, time
R7 = 'Qwen/Qwen2.5-7B-Instruct'
N = '500'
os.makedirs('src/results', exist_ok=True)

def run(cmd, log):
    t0 = time.time()
    with open(log, 'w') as lf:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True); lf.write(line)
        p.wait()
    print(f'\n[{time.time()-t0:.0f}s] rc={p.returncode}')
    return p.returncode

BASE_ARGS = [sys.executable, '-u', 'src/scripts/run_eval.py', '--dataset', 'vimqa', '--limit', N,
             '--reader', 'hf', '--reader-model', R7, '--load-4bit', '--itercomp-llm', 'hf',
             '--scorer', 'dual', '--lam', '0.6', '--max-iter', '5', '--methods', 'itercomp']
RUNS = {
    'E0': ('src/results/evidence_base_500.json', []),
    'E1': ('src/results/evidence_window1_500.json', ['--window', '1']),
    'E2': ('src/results/evidence_entity_500.json', ['--followup-mode', 'entity', '--title-anchor']),
}
for name, (out, extra) in RUNS.items():
    if os.path.exists(out): print('[bỏ qua]', out); continue
    print('=' * 60, '\n', name, extra, flush=True)
    run(BASE_ARGS + extra + ['--out', out], f'{name}.log')

E3 = 'src/results/exp2_recall_500.json'
if os.path.exists(E3): print('[bỏ qua]', E3)
else:
    run([sys.executable, '-u', 'src/scripts/exp2_undiacritised.py', '--limit', N,
         '--reader', 'hf', '--reader-model', R7, '--load-4bit', '--modes', 'keep,strip',
         '--out', E3], 'E3.log')

In [ ]:
# Tổng hợp: recall câu vàng, F1*, tỉ lệ giữ, số vòng; ghép cặp với E0
import json, numpy as np
sys.path.insert(0, 'src')
from itercomp import load_dataset
rows = load_dataset('vimqa', int(N))
def gold_set(r):
    sf = r['supporting_facts']
    have = {(t, i) for t, ss in zip(r['context']['title'], r['context']['sentences'])
            for i, x in enumerate(ss) if x.strip()}
    return set(zip(sf['title'], sf['sent_id'])) & have

def boot(d, B=10000, seed=0):
    d = np.asarray(d, float); rng = np.random.default_rng(seed)
    m = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(B)]
    return d.mean(), np.percentile(m, 2.5), np.percentile(m, 97.5)

stats = {}
for name, (out, _) in RUNS.items():
    pr = json.load(open(out, encoding='utf-8'))['per_row']
    assert [p['question'] for p in pr] == [r['question'] for r in rows[:len(pr)]]
    rec, full, f1, ratio, it = [], [], [], [], []
    for p, r in zip(pr, rows):
        m = p['methods']['itercomp']; g = gold_set(r); sel = {tuple(x) for x in m['selected']}
        rec.append(len(g & sel) / len(g) if g else np.nan); full.append(float(g <= sel))
        f1.append(m['f1_norm'] * 100); ratio.append(m['tokens'] / p['orig_tokens']); it.append(m['iterations'])
    stats[name] = dict(rec=np.array(rec), full=np.array(full), f1=np.array(f1), ratio=np.array(ratio), it=np.array(it))
    s = stats[name]
    print(f"{name}: recall {np.nanmean(s['rec'])*100:5.1f}%  all-gold {s['full'].mean()*100:5.1f}%  "
          f"F1* {s['f1'].mean():5.2f}  ratio {s['ratio'].mean():.3f}  iters {s['it'].mean():.2f}")
for name in ('E1', 'E2'):
    dF = boot(stats[name]['f1'] - stats['E0']['f1'])
    dR = boot(np.nan_to_num(stats[name]['rec'] - stats['E0']['rec']) * 100)
    print(f"{name} - E0: F1* {dF[0]:+.2f} [{dF[1]:+.2f},{dF[2]:+.2f}]  recall {dR[0]:+.1f} [{dR[1]:+.1f},{dR[2]:+.1f}] pts")

e3 = json.load(open(E3, encoding='utf-8'))['detail']
k = np.array([d['gold_recall'] for d in e3['keep']], float); s_ = np.array([d['gold_recall'] for d in e3['strip']], float)
ok = ~np.isnan(k) & ~np.isnan(s_)
dR = boot((s_[ok] - k[ok]) * 100)
print(f"E3 recall keep {np.nanmean(k)*100:.1f}%  strip {np.nanmean(s_)*100:.1f}%  strip-keep {dR[0]:+.1f} [{dR[1]:+.1f},{dR[2]:+.1f}] pts")
json.dump({n: {kk: float(np.nanmean(v)) for kk, v in s.items()} for n, s in stats.items()},
          open('src/results/evidence_summary_500.json', 'w'), indent=1)

In [ ]:
import shutil
BASE = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
for f in ['evidence_base_500.json', 'evidence_window1_500.json', 'evidence_entity_500.json',
          'exp2_recall_500.json', 'evidence_summary_500.json']:
    p = 'src/results/' + f
    if os.path.exists(p): shutil.copy(p, os.path.join(BASE, f)); print('✓', f)